In [ ]:

# Cette version compare directement la question de l'utilisateur avec les différents réponses éventuelles 


import json
from sentence_transformers import SentenceTransformer
import numpy as np

def load_qa_data(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    questions, answers = [], []
    for entry in raw_data:
        for q in entry["questions"]:
            questions.append(q)
            answers.append(entry["answer"])
    return questions, answers

def create_embeddings(texts, model_name='all-MiniLM-L12-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts)
    return model, embeddings

def get_best_answer_from_answers(user_question, answers, embeddings, model, threshold=0.5):
    user_embedding = model.encode([user_question])[0]
    similarities = np.dot(embeddings, user_embedding)
    max_score = similarities.max()

    if max_score < threshold:
        return None, max_score

    best_index = similarities.argmax()
    return answers[best_index], max_score

def get_answers_from_multiple_models(user_question, answers):
    models = [
        "all-MiniLM-L6-v2",
        "gtr-t5-base"
    ]

    results = []

    for model_name in models:
        model, answer_embeddings = create_embeddings(answers, model_name)
        best_answer, score = get_best_answer_from_answers(user_question, answers, answer_embeddings, model)
        results.append({
            'model': model_name,
            'best_answer': best_answer,
            'score': score
        })

    return results


json_path = "data.json"
questions, answers = load_qa_data(json_path)

user_input = input("❓ Pose ta question : ")

results = get_answers_from_multiple_models(user_input, answers)

for result in results:
    print(f"\n Modèle : {result['model']}")
    if result['best_answer']:
        print(f" Réponse (score = {result['score']:.2f}) : {result['best_answer']}")
    else:
        print(f"Aucune réponse pertinente trouvée (score = {result['score']:.2f})")


/home/abdellatifunix/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-04-17 17:25:09.362315: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-17 17:25:09.964874: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744903510.255530    1459 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744903510.339550    1459 cuda_blas.cc:1418] Un

RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.